# Notebook 03 — Phase E+F Complexity, Stacking & Cycle Closure

This notebook is treated as a single integrated entity combining Phase E (complexity/stacking) and Phase F (closed-loop recurrence).

Core outputs (Phase E):
- Escalation by burden
- Augmentation/cycle recurrence metrics
- Comorbidity-stratified hazard
- Window quantification table
- Complexity growth curve

Stage F extension outputs:
- Post-admission response-state transitions (stabilized/partial/nonresponse)
- Re-entry cycle clock reset after escalation events
- Simulated recurrent escalation events with cooldown
- Closed-loop recurrence metrics and transition report

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
DATA_ROOT = PROJECT_ROOT / 'Data'
INTERIM_ROOT = DATA_ROOT / 'interim'
META_ROOT = DATA_ROOT / 'metadata'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook03_phase_e'
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_e'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook03_phase_e'
for d in [FIG_DIR, TABLE_DIR, REPORT_DIR, META_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

phase_b = pd.read_parquet(INTERIM_ROOT / 'instability_phase_b_panel.parquet')
inpatient = pd.read_parquet(DATA_ROOT / 'inpatient_event.parquet')
with open(PROJECT_ROOT / 'Results' / 'tables' / 'notebook02_phase_c' / 'phase_c_probabilistic_checks.json', 'r', encoding='utf-8') as f:
    phase_c_checks = json.load(f)

phase_b = phase_b.sort_values(['patient_id', 'day']).reset_index(drop=True)
phase_b['day'] = phase_b['day'].astype(np.int32)

inpatient_month = inpatient.copy()
inpatient_month['day'] = (inpatient_month['admission_day'].astype(np.int32) // 30) * 30
inpatient_month = inpatient_month[['patient_id', 'day']].drop_duplicates()
inpatient_month['escalation_event'] = 1

panel = phase_b.merge(inpatient_month, on=['patient_id', 'day'], how='left')
panel['escalation_event'] = panel['escalation_event'].fillna(0).astype(np.int8)

beta = phase_c_checks['beta']
panel['month_idx'] = panel.groupby('patient_id').cumcount().astype(np.int16)
panel['log_t'] = np.log1p(panel['month_idx'].astype(np.float32)).astype(np.float32)
x = (
    float(beta['beta_I']) * panel['I_phase_b'].astype(np.float32)
    + float(beta['beta_log_tsm']) * panel['log_t']
    + float(beta['beta_inc']) * panel['increment_total'].astype(np.float32)
    + float(beta['beta_time']) * (panel['month_idx'].astype(np.float32) / 12.0)
)
panel['hazard_prob'] = (1.0 / (1.0 + np.exp(-np.clip(float(beta['beta_0']) + x, -12, 12)))).astype(np.float32)

print('Phase E panel rows:', len(panel))
print('Patients:', panel['patient_id'].nunique())
panel[['patient_id','day','I_phase_b','active_diagnosis_count','active_medication_count','hazard_prob','escalation_event']].head(5)

Phase E panel rows: 4900000
Patients: 100000


,patient_id,day,I_phase_b,active_diagnosis_count,active_medication_count,hazard_prob,escalation_event
0,P000000,0,0.399309,1,1,0.003945,0
1,P000000,30,0.219145,1,1,0.003946,0
2,P000000,60,0.120269,1,1,0.003969,0
3,P000000,90,0.246005,1,1,0.005037,0
4,P000000,120,0.535011,1,2,0.007343,1


In [2]:
spike_threshold = float(panel['I_phase_b'].quantile(0.8))
panel['instability_spike'] = (panel['I_phase_b'] >= spike_threshold).astype(np.int8)

adm = panel.loc[panel['escalation_event'] == 1, ['patient_id', 'day']].copy()
spk = panel.loc[panel['instability_spike'] == 1, ['patient_id', 'day']].copy()

window_records = []
for pid, g in adm.groupby('patient_id'):
    s = spk.loc[spk['patient_id'] == pid, 'day'].to_numpy(dtype=np.int32)
    if len(s) == 0:
        for d in g['day'].to_numpy(dtype=np.int32):
            window_records.append((pid, int(d), np.nan, False))
        continue
    for d in g['day'].to_numpy(dtype=np.int32):
        prev = s[s <= d]
        if len(prev) == 0:
            window_records.append((pid, int(d), np.nan, False))
        else:
            lag_days = int(d - prev.max())
            window_records.append((pid, int(d), lag_days, lag_days <= 180))

window_df = pd.DataFrame(window_records, columns=['patient_id','admission_day','days_since_spike','preceded_within_180d'])
window_df.to_csv(TABLE_DIR / 'phase_e_window_quantification_raw.csv', index=False)

window_summary = {
    'spike_threshold_I': spike_threshold,
    'n_admissions': int(len(window_df)),
    'n_with_observed_window': int(window_df['days_since_spike'].notna().sum()),
    'mean_days_spike_to_admission': float(window_df['days_since_spike'].dropna().mean()) if window_df['days_since_spike'].notna().any() else np.nan,
    'median_days_spike_to_admission': float(window_df['days_since_spike'].dropna().median()) if window_df['days_since_spike'].notna().any() else np.nan,
    'p90_days_spike_to_admission': float(window_df['days_since_spike'].dropna().quantile(0.9)) if window_df['days_since_spike'].notna().any() else np.nan,
    'pct_admissions_preceded_within_180d': float(window_df['preceded_within_180d'].mean()) if len(window_df) else np.nan
}
with open(TABLE_DIR / 'phase_e_window_quantification.json', 'w', encoding='utf-8') as f:
    json.dump(window_summary, f, indent=4)

hist = window_df['days_since_spike'].dropna()
plt.figure(figsize=(8,5))
if len(hist) > 0:
    plt.hist(hist, bins=30)
plt.title('Window Length Distribution (Spike to Admission)')
plt.xlabel('Days')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR / 'phase_e_window_length_distribution.png', dpi=140, bbox_inches='tight')
plt.close()

window_summary

{'spike_threshold_I': 0.37344659566879274,
 'n_admissions': 42773,
 'n_with_observed_window': 33768,
 'mean_days_spike_to_admission': 46.26154939587775,
 'median_days_spike_to_admission': 0.0,
 'p90_days_spike_to_admission': 150.0,
 'pct_admissions_preceded_within_180d': 0.7201271830360274}

In [3]:
adm_days = panel.loc[panel['escalation_event'] == 1, ['patient_id','day']].sort_values(['patient_id','day'])
adm_days['prev_day'] = adm_days.groupby('patient_id')['day'].shift(1)
adm_days['cycle_gap_days'] = (adm_days['day'] - adm_days['prev_day']).astype('float')

cycles_per_patient = adm_days.groupby('patient_id').size().rename('admission_cycles').reset_index()
recurrence_rate = float((cycles_per_patient['admission_cycles'] > 1).mean()) if len(cycles_per_patient) else np.nan
mean_cycles = float(cycles_per_patient['admission_cycles'].mean()) if len(cycles_per_patient) else np.nan
mean_gap = float(adm_days['cycle_gap_days'].dropna().mean()) if adm_days['cycle_gap_days'].notna().any() else np.nan

cycle_metrics = pd.DataFrame([{
    'admitted_patients': int(len(cycles_per_patient)),
    'mean_cycles_per_admitted_patient': mean_cycles,
    'admission_recurrence_rate': recurrence_rate,
    'mean_time_between_cycles_days': mean_gap
}])
cycle_metrics.to_csv(TABLE_DIR / 'phase_e_cycle_recurrence_metrics.csv', index=False)

growth = panel.groupby('day', as_index=False).agg(
    mean_dx=('active_diagnosis_count','mean'),
    mean_med=('active_medication_count','mean'),
    p90_dx=('active_diagnosis_count', lambda s: float(np.quantile(s, 0.90))),
    p90_med=('active_medication_count', lambda s: float(np.quantile(s, 0.90)))
)
growth.to_csv(TABLE_DIR / 'phase_e_complexity_growth_curve.csv', index=False)

plt.figure(figsize=(10,5))
plt.plot(growth['day'], growth['mean_dx'], label='Mean diagnosis count', linewidth=2)
plt.plot(growth['day'], growth['mean_med'], label='Mean medication count', linewidth=2)
plt.plot(growth['day'], growth['p90_dx'], label='P90 diagnosis', linestyle='--')
plt.plot(growth['day'], growth['p90_med'], label='P90 medication', linestyle='--')
plt.title('Complexity Growth Over Time')
plt.xlabel('Day')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'phase_e_complexity_growth_curve.png', dpi=140, bbox_inches='tight')
plt.close()

cycle_metrics

,admitted_patients,mean_cycles_per_admitted_patient,admission_recurrence_rate,mean_time_between_cycles_days
0,42773,1.0,0.0,NaN


In [ ]:
panel['complexity_score'] = (panel['active_diagnosis_count'].astype(float) + panel['active_medication_count'].astype(float)).astype(np.float32)
panel['complexity_quintile'] = pd.qcut(panel['complexity_score'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')

haz_by_q = panel.groupby('complexity_quintile', as_index=False).agg(
    mean_hazard=('hazard_prob','mean'),
    escalation_rate=('escalation_event','mean'),
    mean_dx=('active_diagnosis_count','mean'),
    mean_med=('active_medication_count','mean'),
    n=('patient_id','size')
)
haz_by_q.to_csv(TABLE_DIR / 'phase_e_hazard_by_complexity_quintile.csv', index=False)

comorbidity_bins = pd.cut(panel['active_diagnosis_count'], bins=[0,1,2,3,5,10,100], labels=['1','2','3','4-5','6-10','11+'], include_lowest=True)
haz_by_comorb = panel.groupby(comorbidity_bins, as_index=False, observed=False).agg(
    mean_hazard=('hazard_prob','mean'),
    escalation_rate=('escalation_event','mean'),
    mean_complexity=('complexity_score','mean'),
    n=('patient_id','size')
)
haz_by_comorb.to_csv(TABLE_DIR / 'phase_e_comorbidity_stratified_hazard.csv', index=False)

plot_q = haz_by_q.dropna(subset=['complexity_quintile'])
plt.figure(figsize=(8,5))
plt.plot(plot_q['complexity_quintile'].astype(str), plot_q['mean_hazard'], marker='o', linewidth=2)
plt.title('Hazard by Complexity Quintile')
plt.xlabel('Complexity Quintile')
plt.ylabel('Mean Hazard')
plt.tight_layout()
plt.savefig(FIG_DIR / 'phase_e_hazard_by_complexity_quintile.png', dpi=140, bbox_inches='tight')
plt.close()

summary_lines = [
    'Phase E Complexity & Stacking Summary',
    f"rows: {len(panel)}",
    f"patients: {panel['patient_id'].nunique()}",
    f"mean_cycles_per_admitted_patient: {float(cycle_metrics['mean_cycles_per_admitted_patient'].iloc[0]):.6f}",
    f"admission_recurrence_rate: {float(cycle_metrics['admission_recurrence_rate'].iloc[0]):.6f}",
    f"mean_time_between_cycles_days: {float(cycle_metrics['mean_time_between_cycles_days'].iloc[0]) if pd.notna(cycle_metrics['mean_time_between_cycles_days'].iloc[0]) else np.nan}",
    f"window_pct_preceded_within_180d: {window_summary['pct_admissions_preceded_within_180d']}"
]
with open(REPORT_DIR / 'phase_e_complexity_summary.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))

manifest = {
    'phase': 'E',
    'notebook': '03_phase_e_f_cycle_closure.ipynb',
    'inputs': [
        'Data/interim/instability_phase_b_panel.parquet',
        'Data/inpatient_event.parquet',
        'Results/tables/notebook02_phase_c/phase_c_probabilistic_checks.json'
    ],
    'outputs_tables': [
        'Results/tables/notebook03_phase_e/phase_e_window_quantification.json',
        'Results/tables/notebook03_phase_e/phase_e_cycle_recurrence_metrics.csv',
        'Results/tables/notebook03_phase_e/phase_e_complexity_growth_curve.csv',
        'Results/tables/notebook03_phase_e/phase_e_hazard_by_complexity_quintile.csv',
        'Results/tables/notebook03_phase_e/phase_e_comorbidity_stratified_hazard.csv'
    ],
    'outputs_figures': [
        'Results/figures/notebook03_phase_e/phase_e_window_length_distribution.png',
        'Results/figures/notebook03_phase_e/phase_e_complexity_growth_curve.png',
        'Results/figures/notebook03_phase_e/phase_e_hazard_by_complexity_quintile.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook03_phase_e/phase_e_complexity_summary.txt'
    ]
}
with open(META_ROOT / 'phase_e_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=4)

proof = {
    'window_quantification_table_present': (TABLE_DIR / 'phase_e_window_quantification.json').exists(),
    'cycle_recurrence_metrics_present': (TABLE_DIR / 'phase_e_cycle_recurrence_metrics.csv').exists(),
    'complexity_growth_curve_present': (TABLE_DIR / 'phase_e_complexity_growth_curve.csv').exists(),
    'hazard_by_complexity_quintile_present': (TABLE_DIR / 'phase_e_hazard_by_complexity_quintile.csv').exists(),
    'comorbidity_stratified_hazard_present': (TABLE_DIR / 'phase_e_comorbidity_stratified_hazard.csv').exists(),
    'metadata_manifest_present': (META_ROOT / 'phase_e_manifest.json').exists()
}
with open(REPORT_DIR / 'phase_e_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof, 'window_summary': window_summary}, f, indent=4)

print('Phase E outputs generated')
for k, v in proof.items():
    print('-', k, ':', v)

haz_by_q

Phase E outputs generated
- window_quantification_table_present : True
- cycle_recurrence_metrics_present : True
- complexity_growth_curve_present : True
- hazard_by_complexity_quintile_present : True
- comorbidity_stratified_hazard_present : True
- metadata_manifest_present : True


,complexity_quintile,mean_hazard,escalation_rate,mean_dx,mean_med,n
0,Q1,0.004794,0.010596,1.041978,1.469186,1143026
1,Q2,0.006412,0.009373,1.277587,3.209768,1088818
2,Q3,0.008230,0.008484,1.646063,4.835660,973628
3,Q4,0.008919,0.007503,2.031881,6.428492,779684
4,Q5,0.009441,0.006937,2.654430,8.339325,914844


## Stage F — Closed-Loop Recurrence and Re-Entry Dynamics

This stage closes the cycle by applying post-admission response states, resetting cycle clocks after each escalation event, and simulating recurrent escalations under a non-deterministic policy.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
FIG_DIR_F = PROJECT_ROOT / 'Results' / 'figures' / 'notebook03_phase_f'
TABLE_DIR_F = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_f'
REPORT_DIR_F = PROJECT_ROOT / 'Results' / 'reports' / 'notebook03_phase_f'
META_ROOT = PROJECT_ROOT / 'Data' / 'metadata'
for d in [FIG_DIR_F, TABLE_DIR_F, REPORT_DIR_F, META_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

panel_f = panel.copy().sort_values(['patient_id', 'day']).reset_index(drop=True)
panel_f['baseline_admission_event'] = panel_f['escalation_event'].astype(np.int8)
panel_f['complexity_score'] = (
    panel_f['active_diagnosis_count'].astype(np.float32) + panel_f['active_medication_count'].astype(np.float32)
).astype(np.float32)

# --- Response state assignment at each baseline admission ---
adm_rows = panel_f.loc[panel_f['baseline_admission_event'] == 1, [
    'patient_id', 'day', 'I_phase_b', 'active_diagnosis_count', 'active_medication_count', 'complexity_score'
 ]].copy()

logit_nonresponse = (
    -2.2
    + 0.22 * adm_rows['active_diagnosis_count'].astype(np.float32)
    + 0.12 * adm_rows['active_medication_count'].astype(np.float32)
    + 0.90 * adm_rows['I_phase_b'].astype(np.float32)
)
p_non = 1.0 / (1.0 + np.exp(-np.clip(logit_nonresponse, -8, 8)))
p_stab = np.clip(0.62 - 0.38 * p_non, 0.20, 0.70)
p_partial = np.clip(1.0 - p_stab - p_non, 0.10, 0.65)
norm = (p_stab + p_partial + p_non).astype(np.float32)
p_stab = (p_stab / norm).astype(np.float32)
p_partial = (p_partial / norm).astype(np.float32)

u = np.random.rand(len(adm_rows)).astype(np.float32)
adm_rows['response_state'] = np.where(
    u < p_stab,
    'stabilized',
    np.where(u < (p_stab + p_partial), 'partial_response', 'nonresponse')
)

adm_rows['response_multiplier'] = adm_rows['response_state'].map({
    'stabilized': 0.62,
    'partial_response': 0.82,
    'nonresponse': 1.08
}).astype(np.float32)
adm_rows['instability_shift'] = adm_rows['response_state'].map({
    'stabilized': -0.14,
    'partial_response': -0.05,
    'nonresponse': 0.03
}).astype(np.float32)

panel_f = panel_f.merge(
    adm_rows[['patient_id', 'day', 'response_state', 'response_multiplier', 'instability_shift']],
    on=['patient_id', 'day'],
    how='left'
 )

panel_f['response_state_active'] = panel_f.groupby('patient_id')['response_state'].ffill()
panel_f['response_state_active'] = panel_f['response_state_active'].fillna('pre_admission')
panel_f['response_multiplier_active'] = panel_f.groupby('patient_id')['response_multiplier'].ffill().fillna(1.0).astype(np.float32)
panel_f['instability_shift_active'] = panel_f.groupby('patient_id')['instability_shift'].ffill().fillna(0.0).astype(np.float32)

panel_f['last_admission_day'] = panel_f['day'].where(panel_f['baseline_admission_event'] == 1)
panel_f['last_admission_day'] = panel_f.groupby('patient_id')['last_admission_day'].ffill()
panel_f['months_since_admission'] = np.where(
    panel_f['last_admission_day'].notna(),
    ((panel_f['day'] - panel_f['last_admission_day']) / 30.0).clip(lower=0),
    np.nan
).astype(np.float32)

effect_decay = np.where(
    panel_f['months_since_admission'].notna(),
    np.exp(-panel_f['months_since_admission'].astype(np.float32) / 6.0),
    0.0
).astype(np.float32)

adj_multiplier = (1.0 - (1.0 - panel_f['response_multiplier_active'].astype(np.float32)) * effect_decay).astype(np.float32)
panel_f['I_stage_f_base'] = np.maximum(
    0.0,
    panel_f['I_phase_b'].astype(np.float32) + panel_f['instability_shift_active'].astype(np.float32) * effect_decay
).astype(np.float32)
panel_f['hazard_prob_stage_f_base'] = np.clip(
    panel_f['hazard_prob'].astype(np.float32) * adj_multiplier,
    1e-6,
    0.35
).astype(np.float32)

# --- Simulate recurrent escalation events (closed-loop) ---
pid = panel_f['patient_id'].to_numpy()
base_event = panel_f['baseline_admission_event'].to_numpy(dtype=np.int8)
p_base = panel_f['hazard_prob_stage_f_base'].to_numpy(dtype=np.float32)
months_since = panel_f['months_since_admission'].to_numpy(dtype=np.float32)
resp = panel_f['response_state_active'].to_numpy()

resp_boost = np.where(
    resp == 'nonresponse', 1.22,
    np.where(resp == 'partial_response', 1.08, np.where(resp == 'stabilized', 0.92, 1.00))
).astype(np.float32)
time_boost = (1.0 + 0.04 * np.minimum(np.nan_to_num(months_since, nan=0.0), 12.0)).astype(np.float32)
p_recur = np.clip(p_base * resp_boost * time_boost, 1e-6, 0.30).astype(np.float32)

stage_f_event = np.zeros(len(panel_f), dtype=np.int8)
cooldown_months = 0
has_had_admission = False
prev_pid = pid[0]

for i in range(len(panel_f)):
    if pid[i] != prev_pid:
        prev_pid = pid[i]
        cooldown_months = 0
        has_had_admission = False

    if base_event[i] == 1:
        stage_f_event[i] = 1
        has_had_admission = True
        cooldown_months = 2
        continue

    if cooldown_months > 0:
        cooldown_months -= 1
        continue

    if not has_had_admission:
        continue

    if np.random.rand() < p_recur[i]:
        stage_f_event[i] = 1
        cooldown_months = 2

panel_f['stage_f_escalation_event'] = stage_f_event.astype(np.int8)

# --- Re-entry cycle clock reset after every Stage F escalation event ---
panel_f['cycle_id_stage_f'] = panel_f.groupby('patient_id')['stage_f_escalation_event'].cumsum().astype(np.int16)
panel_f['months_since_cycle_start'] = panel_f.groupby(['patient_id', 'cycle_id_stage_f']).cumcount().astype(np.int16)
panel_f['log_tsm_stage_f'] = np.log1p(panel_f['months_since_cycle_start'].astype(np.float32)).astype(np.float32)

beta = phase_c_checks['beta']
x_f = (
    float(beta['beta_I']) * panel_f['I_stage_f_base'].astype(np.float32)
    + float(beta['beta_log_tsm']) * panel_f['log_tsm_stage_f']
    + float(beta['beta_inc']) * panel_f['increment_total'].astype(np.float32)
    + float(beta['beta_time']) * (panel_f['months_since_cycle_start'].astype(np.float32) / 12.0)
)
panel_f['hazard_prob_stage_f'] = (
    1.0 / (1.0 + np.exp(-np.clip(float(beta['beta_0']) + x_f, -12, 12)))
).astype(np.float32)

# --- Stage F tables ---
response_mix = (
    adm_rows['response_state'].value_counts(dropna=False)
    .rename_axis('response_state')
    .reset_index(name='n')
)
response_mix['share'] = (response_mix['n'] / max(len(adm_rows), 1)).astype(np.float32)
response_mix.to_csv(TABLE_DIR_F / 'phase_f_admission_response_profile.csv', index=False)

adm_f = panel_f.loc[panel_f['stage_f_escalation_event'] == 1, ['patient_id', 'day']].sort_values(['patient_id', 'day']).copy()
adm_f['prev_day'] = adm_f.groupby('patient_id')['day'].shift(1)
adm_f['cycle_gap_days'] = (adm_f['day'] - adm_f['prev_day']).astype(float)
cycles_per_patient_f = adm_f.groupby('patient_id').size().rename('admission_cycles').reset_index()

recurrence_rate_f = float((cycles_per_patient_f['admission_cycles'] > 1).mean()) if len(cycles_per_patient_f) else np.nan
mean_cycles_f = float(cycles_per_patient_f['admission_cycles'].mean()) if len(cycles_per_patient_f) else np.nan
mean_gap_f = float(adm_f['cycle_gap_days'].dropna().mean()) if adm_f['cycle_gap_days'].notna().any() else np.nan

cycle_metrics_f = pd.DataFrame([{
    'admitted_patients': int(len(cycles_per_patient_f)),
    'total_escalation_events': int(len(adm_f)),
    'mean_cycles_per_admitted_patient': mean_cycles_f,
    'admission_recurrence_rate': recurrence_rate_f,
    'mean_time_between_cycles_days': mean_gap_f
}])
cycle_metrics_f.to_csv(TABLE_DIR_F / 'phase_f_cycle_recurrence_metrics.csv', index=False)

cycle_hazard = panel_f.groupby('months_since_cycle_start', as_index=False).agg(
    mean_hazard_stage_f=('hazard_prob_stage_f', 'mean'),
    p90_hazard_stage_f=('hazard_prob_stage_f', lambda s: float(np.quantile(s, 0.90))),
    n=('patient_id', 'size')
)
cycle_hazard.to_csv(TABLE_DIR_F / 'phase_f_cycle_hazard_trajectory.csv', index=False)

transition_summary = {
    'n_rows': int(len(panel_f)),
    'n_patients': int(panel_f['patient_id'].nunique()),
    'baseline_admissions': int(panel_f['baseline_admission_event'].sum()),
    'stage_f_total_escalations': int(panel_f['stage_f_escalation_event'].sum()),
    'additional_recurrent_escalations': int(panel_f['stage_f_escalation_event'].sum() - panel_f['baseline_admission_event'].sum()),
    'mean_cycles_per_admitted_patient': mean_cycles_f,
    'admission_recurrence_rate': recurrence_rate_f,
    'mean_time_between_cycles_days': mean_gap_f
}
with open(TABLE_DIR_F / 'phase_f_transition_dynamics_summary.json', 'w', encoding='utf-8') as f:
    json.dump(transition_summary, f, indent=4)

panel_f[[
    'patient_id','day','I_phase_b','I_stage_f_base','hazard_prob','hazard_prob_stage_f_base',
    'hazard_prob_stage_f','baseline_admission_event','stage_f_escalation_event',
    'cycle_id_stage_f','months_since_cycle_start','response_state_active'
 ]].to_parquet(TABLE_DIR_F / 'phase_f_closed_loop_panel.parquet', index=False)

# --- Stage F figures ---
plt.figure(figsize=(7,5))
plt.bar(response_mix['response_state'].astype(str), response_mix['share'])
plt.title('Stage F Post-Admission Response Mix')
plt.xlabel('Response state')
plt.ylabel('Share')
plt.tight_layout()
plt.savefig(FIG_DIR_F / 'phase_f_response_mix.png', dpi=140, bbox_inches='tight')
plt.close()

gap_hist = adm_f['cycle_gap_days'].dropna()
plt.figure(figsize=(8,5))
if len(gap_hist) > 0:
    plt.hist(gap_hist, bins=30)
plt.title('Stage F Recurrence Gap Distribution')
plt.xlabel('Days between escalation events')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR_F / 'phase_f_recurrence_gap_distribution.png', dpi=140, bbox_inches='tight')
plt.close()

traj_plot = cycle_hazard[cycle_hazard['months_since_cycle_start'] <= 24]
plt.figure(figsize=(9,5))
plt.plot(traj_plot['months_since_cycle_start'], traj_plot['mean_hazard_stage_f'], label='Mean hazard', linewidth=2)
plt.plot(traj_plot['months_since_cycle_start'], traj_plot['p90_hazard_stage_f'], label='P90 hazard', linestyle='--')
plt.title('Stage F Hazard Trajectory After Cycle Reset')
plt.xlabel('Months since cycle start')
plt.ylabel('Hazard probability')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR_F / 'phase_f_cycle_hazard_trajectory.png', dpi=140, bbox_inches='tight')
plt.close()

summary_lines_f = [
    'Stage F Closed-Loop Recurrence Summary',
    f"rows: {len(panel_f)}",
    f"patients: {panel_f['patient_id'].nunique()}",
    f"baseline_admissions: {int(panel_f['baseline_admission_event'].sum())}",
    f"stage_f_total_escalations: {int(panel_f['stage_f_escalation_event'].sum())}",
    f"additional_recurrent_escalations: {int(panel_f['stage_f_escalation_event'].sum() - panel_f['baseline_admission_event'].sum())}",
    f"mean_cycles_per_admitted_patient: {mean_cycles_f:.6f}",
    f"admission_recurrence_rate: {recurrence_rate_f:.6f}",
    f"mean_time_between_cycles_days: {mean_gap_f if pd.notna(mean_gap_f) else np.nan}"
]
with open(REPORT_DIR_F / 'phase_f_closed_loop_summary.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines_f))

manifest_f = {
    'phase': 'F',
    'notebook': '03_phase_e_f_cycle_closure.ipynb',
    'inputs': [
        'Data/interim/instability_phase_b_panel.parquet',
        'Data/inpatient_event.parquet',
        'Results/tables/notebook02_phase_c/phase_c_probabilistic_checks.json'
    ],
    'outputs_tables': [
        'Results/tables/notebook03_phase_f/phase_f_admission_response_profile.csv',
        'Results/tables/notebook03_phase_f/phase_f_cycle_recurrence_metrics.csv',
        'Results/tables/notebook03_phase_f/phase_f_cycle_hazard_trajectory.csv',
        'Results/tables/notebook03_phase_f/phase_f_transition_dynamics_summary.json',
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet'
    ],
    'outputs_figures': [
        'Results/figures/notebook03_phase_f/phase_f_response_mix.png',
        'Results/figures/notebook03_phase_f/phase_f_recurrence_gap_distribution.png',
        'Results/figures/notebook03_phase_f/phase_f_cycle_hazard_trajectory.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook03_phase_f/phase_f_closed_loop_summary.txt'
    ]
}
with open(META_ROOT / 'phase_f_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_f, f, indent=4)

proof_f = {
    'post_admission_response_states_assigned': bool(len(adm_rows) > 0),
    'reentry_cycle_clock_resets_implemented': True,
    'recurrent_escalation_simulated': bool(panel_f['stage_f_escalation_event'].sum() >= panel_f['baseline_admission_event'].sum()),
    'recurrence_metrics_generated': (TABLE_DIR_F / 'phase_f_cycle_recurrence_metrics.csv').exists(),
    'transition_summary_generated': (TABLE_DIR_F / 'phase_f_transition_dynamics_summary.json').exists(),
    'closed_loop_panel_generated': (TABLE_DIR_F / 'phase_f_closed_loop_panel.parquet').exists(),
    'manifest_generated': (META_ROOT / 'phase_f_manifest.json').exists()
}
with open(REPORT_DIR_F / 'phase_f_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof_f, 'summary': transition_summary}, f, indent=4)

print('Stage F outputs generated')
for k, v in proof_f.items():
    print('-', k, ':', v)

cycle_metrics_f

In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook03'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)